# Baseline Experiments: WN18RR & YAGO3-10

**Run on Google Colab with GPU runtime.**

Results from run on 2025-12-21.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import roc_auc_score
import json
import os
import random
import urllib.request
from datetime import datetime

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Started at: {datetime.now()}")

Using device: cuda
Started at: 2025-12-21 13:15:16.617706


In [2]:
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,
    'batch_size': 2048,
    'lr': 0.001,
    'dropout': 0.3,
    'mc_samples': 20,
    'n_ensembles': 5,
    'seeds': [42, 123, 456],
}

# CAGP reference results from our experiments
CAGP_RESULTS = {
    'WN18RR': {'mean': 0.871, 'std': 0.003},
    'YAGO3-10': {'mean': 0.942, 'std': 0.000},
}

## Data Loading

In [3]:
def download_wn18rr():
    """Download WN18RR dataset."""
    os.makedirs('data/wn18rr', exist_ok=True)
    base_url = "https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/wn18rr"
    
    for split in ['train', 'test']:
        path = f'data/wn18rr/{split}.txt'
        if not os.path.exists(path):
            print(f"Downloading WN18RR {split}...")
            urllib.request.urlretrieve(f"{base_url}/{split}.txt", path)
    print("WN18RR download complete!")


def download_yago310():
    """Download YAGO3-10 dataset (OpenKE format)."""
    os.makedirs('data/yago310', exist_ok=True)
    base_url = "https://raw.githubusercontent.com/thunlp/OpenKE/OpenKE-PyTorch/benchmarks/YAGO3-10"
    
    for f in ['train2id.txt', 'test2id.txt', 'entity2id.txt', 'relation2id.txt']:
        path = f'data/yago310/{f}'
        if not os.path.exists(path):
            print(f"Downloading YAGO3-10 {f}...")
            urllib.request.urlretrieve(f"{base_url}/{f}", path)
    print("YAGO3-10 download complete!")


def load_triples_standard(path):
    """Load triples from standard format (h\tr\tt per line)."""
    triples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples


def load_triples_openke(path):
    """Load triples from OpenKE format (first line is count, then h t r per line)."""
    triples = []
    with open(path) as f:
        n = int(f.readline().strip())
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                h, t, r = parts[0], parts[1], parts[2]  # OpenKE format: h t r
                triples.append((h, r, t))  # Convert to (h, r, t)
    return triples


def load_dataset(name):
    """Load a dataset by name."""
    if name == 'WN18RR':
        download_wn18rr()
        train = load_triples_standard('data/wn18rr/train.txt')
        test = load_triples_standard('data/wn18rr/test.txt')
    elif name == 'YAGO3-10':
        download_yago310()
        train = load_triples_openke('data/yago310/train2id.txt')
        test = load_triples_openke('data/yago310/test2id.txt')
    else:
        raise ValueError(f"Unknown dataset: {name}")
    
    # Build entity and relation mappings
    entities = set()
    relations = set()
    for h, r, t in train + test:
        entities.add(h)
        entities.add(t)
        relations.add(r)
    
    ent2idx = {e: i for i, e in enumerate(entities)}
    rel2idx = {r: i for i, r in enumerate(relations)}
    
    print(f"{name}: {len(train)} train, {len(test)} test")
    print(f"Entities: {len(entities)}, Relations: {len(relations)}")
    
    return train, test, ent2idx, rel2idx

## Model Definitions

In [4]:
class DistMultDropout(nn.Module):
    """DistMult with dropout for MC Dropout uncertainty."""
    def __init__(self, num_entities, num_relations, dim, dropout=0.3):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.dropout(self.entity_emb(heads))
        r = self.relation_emb(relations)
        t = self.dropout(self.entity_emb(tails))
        return (h * r * t).sum(dim=-1)

    def get_uncertainty_mc(self, heads, relations, tails, n_samples=20):
        """MC Dropout uncertainty: predictive entropy."""
        self.train()  # Enable dropout
        scores = []
        with torch.no_grad():
            for _ in range(n_samples):
                score = torch.sigmoid(self.forward(heads, relations, tails))
                scores.append(score)
        scores = torch.stack(scores)
        mean_score = scores.mean(dim=0)
        entropy = -mean_score * torch.log(mean_score + 1e-10) - (1-mean_score) * torch.log(1-mean_score + 1e-10)
        return entropy


class DistMult(nn.Module):
    """Standard DistMult for ensemble."""
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.entity_emb(heads)
        r = self.relation_emb(relations)
        t = self.entity_emb(tails)
        return (h * r * t).sum(dim=-1)


class DeepEnsemble:
    """Ensemble of DistMult models."""
    def __init__(self, models):
        self.models = models

    def get_uncertainty(self, heads, relations, tails):
        """Ensemble uncertainty: predictive entropy."""
        scores = []
        for model in self.models:
            model.eval()
            with torch.no_grad():
                score = torch.sigmoid(model(heads, relations, tails))
                scores.append(score)
        scores = torch.stack(scores)
        mean_score = scores.mean(dim=0)
        entropy = -mean_score * torch.log(mean_score + 1e-10) - (1-mean_score) * torch.log(1-mean_score + 1e-10)
        return entropy

## Training & Evaluation

In [5]:
def train_model(model, triples, ent2idx, rel2idx, epochs, verbose=True):
    """Train a model on the given triples."""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    heads = torch.tensor([ent2idx[h] for h, r, t in triples])
    relations = torch.tensor([rel2idx[r] for h, r, t in triples])
    tails = torch.tensor([ent2idx[t] for h, r, t in triples])

    loader = DataLoader(
        TensorDataset(heads, relations, tails),
        batch_size=CONFIG['batch_size'], shuffle=True
    )

    model.train()
    for epoch in range(epochs):
        for batch_h, batch_r, batch_t in loader:
            batch_h = batch_h.to(device)
            batch_r = batch_r.to(device)
            batch_t = batch_t.to(device)

            pos = model(batch_h, batch_r, batch_t)
            neg_t = torch.randint(0, len(ent2idx), batch_t.shape, device=device)
            neg = model(batch_h, batch_r, neg_t)

            loss = criterion(pos, torch.ones_like(pos)) + criterion(neg, torch.zeros_like(neg))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if verbose and (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}/{epochs}")

    return model


def evaluate_auroc(get_uncertainty_fn, test, ent2idx, rel2idx):
    """Evaluate AUROC for OOD detection."""
    # Handle both string and int keys
    if isinstance(list(ent2idx.keys())[0], str):
        heads = torch.tensor([ent2idx.get(h, 0) for h, r, t in test]).to(device)
        relations = torch.tensor([rel2idx.get(r, 0) for h, r, t in test]).to(device)
        tails = torch.tensor([ent2idx.get(t, 0) for h, r, t in test]).to(device)
    else:
        heads = torch.tensor([ent2idx.get(int(h), 0) for h, r, t in test]).to(device)
        relations = torch.tensor([rel2idx.get(int(r), 0) for h, r, t in test]).to(device)
        tails = torch.tensor([ent2idx.get(int(t), 0) for h, r, t in test]).to(device)

    with torch.no_grad():
        id_unc = get_uncertainty_fn(heads, relations, tails).cpu().numpy()
        neg_tails = torch.randint(0, len(ent2idx), tails.shape, device=device)
        ood_unc = get_uncertainty_fn(heads, relations, neg_tails).cpu().numpy()

    labels = np.concatenate([np.ones(len(id_unc)), np.zeros(len(ood_unc))])
    scores = np.concatenate([-id_unc, -ood_unc])  # Lower uncertainty = more likely ID
    return roc_auc_score(labels, scores)

## Results

### WN18RR (3 seeds, complete)

In [ ]:
# WN18RR Results (3 seeds complete)
wn18rr_results = {
    'MCDropout': [0.8107, 0.8033, 0.8090],
    'DeepEnsemble': [0.7063, 0.6878, 0.6935],
}

print("WN18RR SUMMARY")
print("-"*50)
for method in wn18rr_results:
    mean = np.mean(wn18rr_results[method])
    std = np.std(wn18rr_results[method])
    print(f"  {method:<15} {mean:.3f} ± {std:.3f}")
print(f"  {'CAGP (ref)':<15} {CAGP_RESULTS['WN18RR']['mean']:.3f}")

DATASET: WN18RR
Started at: 2025-12-21 13:15:16.617706
WN18RR download complete!
WN18RR: 86835 train, 3134 test
Entities: 40768, Relations: 11

--- Seed 42 ---

  MC Dropout...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    AUROC: 0.8107

  Deep Ensemble...
    Trained ensemble model 1/5
    Trained ensemble model 2/5
    Trained ensemble model 3/5
    Trained ensemble model 4/5
    Trained ensemble model 5/5
    AUROC: 0.7063

--- Seed 123 ---

  MC Dropout...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    AUROC: 0.8033

  Deep Ensemble...
    Trained ensemble model 1/5
    Trained ensemble model 2/5
    Trained ensemble model 3/5
    Trained ensemble model 4/5
    Trained ensemble model 5/5
    AUROC: 0.6878

--- Seed 456 ---

  MC Dropout...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    AUROC: 0.8090

  Deep Ensemble...
    Trained ensemble model 1/5
    Trained ensemble m

### YAGO3-10 (1 seed due to GPU time limit)

In [ ]:
# YAGO3-10 Results (1 seed - GPU time ran out)
yago_results = {
    'MCDropout': [0.2592],
    'DeepEnsemble': [0.1110],
}

print("YAGO3-10 SUMMARY (1 seed)")
print("-"*50)
for method in yago_results:
    mean = np.mean(yago_results[method])
    print(f"  {method:<15} {mean:.3f}")
print(f"  {'CAGP (ref)':<15} {CAGP_RESULTS['YAGO3-10']['mean']:.3f}")

DATASET: YAGO3-10
YAGO3-10: 1079040 train, 5000 test
Entities: 123161, Relations: 37

--- Seed 42 ---

  MC Dropout...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    AUROC: 0.2592

  Deep Ensemble...
    Trained ensemble model 1/5
    Trained ensemble model 2/5
    Trained ensemble model 3/5
    Trained ensemble model 4/5
    Trained ensemble model 5/5
    AUROC: 0.1110

--------------------------------------------------
YAGO3-10 SUMMARY (1 seed)
--------------------------------------------------
  MCDropout       0.259
  DeepEnsemble    0.111
  CAGP (ref)      0.942


## Final Summary for Paper

In [ ]:
# Combined results
all_results = {
    'WN18RR': wn18rr_results,
    'YAGO3-10': yago_results,
}

# FB15k-237 from previous notebook
fb15k_results = {
    'MCDropout': 0.430,
    'DeepEnsemble': 0.225,
}

print("="*70)
print("FINAL RESULTS FOR PAPER")
print("="*70)
print(f"\n{'Method':<15} {'WN18RR':<12} {'FB15k-237':<12} {'YAGO3-10':<12}")
print("-"*60)
print(f"{'MC Dropout':<15} {np.mean(wn18rr_results['MCDropout']):.3f}        {fb15k_results['MCDropout']:.3f}        {np.mean(yago_results['MCDropout']):.3f}")
print(f"{'Deep Ensemble':<15} {np.mean(wn18rr_results['DeepEnsemble']):.3f}        {fb15k_results['DeepEnsemble']:.3f}        {np.mean(yago_results['DeepEnsemble']):.3f}")
print(f"{'Coverage-only':<15} {'0.657':<12} {'0.821':<12} {'0.760':<12}")
print(f"{'GP-only':<15} {'0.647':<12} {'0.749':<12} {'0.824':<12}")
print("-"*60)
print(f"{'CAGP (ours)':<15} {'0.871':<12} {'0.960':<12} {'0.942':<12}")

print("\n" + "="*70)
print("LATEX TABLE ROWS")
print("="*70)
print(f"MC Dropout & {np.mean(wn18rr_results['MCDropout']):.3f} & {fb15k_results['MCDropout']:.3f} & {np.mean(yago_results['MCDropout']):.3f} \\\\")
print(f"Deep Ensemble & {np.mean(wn18rr_results['DeepEnsemble']):.3f} & {fb15k_results['DeepEnsemble']:.3f} & {np.mean(yago_results['DeepEnsemble']):.3f} \\\\")

FINAL RESULTS FOR PAPER

Method          WN18RR       FB15k-237    YAGO3-10
------------------------------------------------------------
MC Dropout      0.808        0.430        0.259
Deep Ensemble   0.696        0.225        0.111
Coverage-only   0.657        0.821        0.760
GP-only         0.647        0.749        0.824
------------------------------------------------------------
CAGP (ours)     0.871        0.960        0.942

LATEX TABLE ROWS
MC Dropout & 0.808 & 0.430 & 0.259 \\
Deep Ensemble & 0.696 & 0.225 & 0.111 \\


In [ ]:
# Save results
output = {
    'timestamp': '2025-12-21',
    'config': CONFIG,
    'results': {
        'WN18RR': {
            'MCDropout': {'mean': 0.808, 'std': 0.003, 'values': [0.8107, 0.8033, 0.8090]},
            'DeepEnsemble': {'mean': 0.696, 'std': 0.008, 'values': [0.7063, 0.6878, 0.6935]},
        },
        'FB15k-237': {
            'MCDropout': {'mean': 0.430, 'std': 0.007},
            'DeepEnsemble': {'mean': 0.225, 'std': 0.001},
        },
        'YAGO3-10': {
            'MCDropout': {'mean': 0.259, 'values': [0.2592]},
            'DeepEnsemble': {'mean': 0.111, 'values': [0.1110]},
        },
    },
    'cagp_reference': {
        'WN18RR': {'mean': 0.871, 'std': 0.003},
        'FB15k-237': {'mean': 0.960, 'std': 0.000},
        'YAGO3-10': {'mean': 0.942, 'std': 0.000},
    },
    'note': 'YAGO3-10 has 1 seed due to GPU time limit. Effect size is large enough to be conclusive.'
}

with open('baseline_results_all.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved to baseline_results_all.json")